# Drag Diffusion — Colab Demo

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better).

Two ways to load the code:
- **Option A (fastest right now):** zip the project locally, upload to Drive, mount below.
- **Option B:** push to GitHub, then use the clone cell.

Run whichever option applies, skip the other.

In [ ]:
# Install dependencies (runs once per runtime)
!pip install -q diffusers>=0.21 transformers>=4.35 accelerate safetensors>=0.3.1 Pillow numpy matplotlib torchvision

## Option A — Upload zip from local machine (no GitHub needed)

On your Mac, run:
```bash
cd ~/Drag-Diffusion && zip -r drag_diffusion.zip . -x 'venv/*' '__pycache__/*' '*.pyc' 'data/*'
```
Then upload `drag_diffusion.zip` to Google Drive and run the cell below.

In [ ]:
# OPTION A: Mount Drive and unzip
from google.colab import drive
drive.mount('/content/drive')

# Update this path to wherever you saved the zip in Drive
ZIP_PATH = '/content/drive/MyDrive/drag_diffusion.zip'

import zipfile, os
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/Drag-Diffusion')

os.chdir('/content/Drag-Diffusion')
print('Working directory:', os.getcwd())

## Option B — Clone from GitHub

Only use this after you've pushed the repo to GitHub.

In [ ]:
# OPTION B: Clone from GitHub (skip if you used Option A)
GITHUB_USER = 'YOUR_GITHUB_USERNAME'  # <-- edit this
REPO_NAME   = 'Drag-Diffusion'        # <-- edit if different

import os
!git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git /content/Drag-Diffusion
os.chdir('/content/Drag-Diffusion')
print('Working directory:', os.getcwd())

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from utils.image_utils import get_device
from pipeline.relocation_pipeline import ObjectRelocationPipeline
from eval.perceptual_loss import VGGPerceptualLoss
from eval.visualize import make_comparison_figure

device = get_device()
print(f'Device: {device}')  # should say cuda

In [ ]:
# Downloads Lykon/dreamshaper-8 on first run (~2 GB) — cached after that
pipe = ObjectRelocationPipeline(device=device, local_files_only=False)
loss_fn = VGGPerceptualLoss(device=device)
print('Pipeline ready.')

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert('RGB')
print(f'Image size: {image.size}')
plt.imshow(image); plt.title('Your image'); plt.axis('off'); plt.show()

In [ ]:
H, W = image.size[1], image.size[0]

# Edit these boxes to match your image (pixels: left, top, right, bottom)
src_box = (50,  100, 200, 300)   # where the object IS
tgt_box = (350, 100, 500, 300)   # where you want it

src_arr = np.zeros((H, W), dtype=np.uint8)
tgt_arr = np.zeros((H, W), dtype=np.uint8)
src_arr[src_box[1]:src_box[3], src_box[0]:src_box[2]] = 255
tgt_arr[tgt_box[1]:tgt_box[3], tgt_box[0]:tgt_box[2]] = 255
source_mask = Image.fromarray(src_arr)
target_mask = Image.fromarray(tgt_arr)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image);       axes[0].set_title('Image');       axes[0].axis('off')
axes[1].imshow(source_mask, cmap='gray'); axes[1].set_title('Source mask'); axes[1].axis('off')
axes[2].imshow(target_mask, cmap='gray'); axes[2].set_title('Target mask'); axes[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
prompt = 'a photo of the scene'  # describe the scene — edit this

print('Running baseline (SDEdit, fresh noise)...')
baseline, composite = pipe(
    image, prompt, source_mask, target_mask,
    use_noise_shift=False, seed=42, num_inference_steps=50, sdedit_strength=0.7,
)

print('Running ours (DDPM noise shift)...')
ours, _ = pipe(
    image, prompt, source_mask, target_mask,
    use_noise_shift=True, seed=42, num_inference_steps=50, sdedit_strength=0.7,
)
print('Done.')

In [ ]:
from utils.image_utils import pil_to_tensor

def obj_crop(img, mask, pad=10):
    arr = np.array(mask.convert('L')) > 127
    ys, xs = np.where(arr)
    if len(ys) == 0:
        return img
    y0, y1 = max(0, ys.min()-pad), ys.max()+pad
    x0, x1 = max(0, xs.min()-pad), xs.max()+pad
    return img.crop((x0, y0, x1, y1)).resize((256, 256))

src_t  = pil_to_tensor(obj_crop(image,    source_mask), device)
bl_t   = pil_to_tensor(obj_crop(baseline, target_mask), device)
ours_t = pil_to_tensor(obj_crop(ours,     target_mask), device)

bl_score   = loss_fn(src_t, bl_t)
ours_score = loss_fn(src_t, ours_t)

print('Perceptual dist (lower = better texture preservation):')
print(f'  Baseline : {bl_score:.4f}')
print(f'  Ours     : {ours_score:.4f}  {"✓ better" if ours_score < bl_score else "✗ worse"}')

make_comparison_figure(image, baseline, ours, 'results.png',
                       title=prompt,
                       scores={'baseline': bl_score, 'ours': ours_score})

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img, t in zip(axes,
                      [image, composite, baseline, ours],
                      ['Original', 'Composite (copy-paste)', 'Baseline', 'Ours']):
    ax.imshow(img); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Optional: download results.png
from google.colab import files
files.download('results.png')